In [2]:
import pandas as pd
import numpy as np

input_file = "dsas_turbine_bearing_vib_g11_01.csv"
output_file = "second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx"

# مرحله 1: خواندن فایل CSV
df = pd.read_csv(input_file)

# مرحله 2: استخراج AssetIDهای یکتا
unique_ids = df['AssetID'].unique().tolist()

# مرحله 3: ساخت دیتافریم خروجی با ستون‌های مورد نظر
columns = ['TimeStamp'] + [f'AssetID_{uid}' for uid in unique_ids]
output_df = pd.DataFrame(columns=columns)

# مرحله 4 تا 8: پردازش تکراری تا خالی شدن دیتافریم اصلی
while not df.empty and unique_ids:
    main_id = unique_ids[0]
    main_subset = df[df['AssetID'] == main_id]

    if main_subset.empty:
        unique_ids.pop(0)
        continue

    # مرحله 5: گرفتن اولین ردیف از AssetID اصلی
    main_row = main_subset.iloc[0]
    main_ts = main_row['TimeStamp']
    main_value = main_row['Value']
    used_indices = [main_row.name]

    # مرحله 6: پیدا کردن نزدیک‌ترین TimeStamp برای سایر AssetIDها
    row_data = {'TimeStamp': main_ts, f'AssetID_{main_id}': main_value}

    for other_id in unique_ids[1:]:
        subset = df[df['AssetID'] == other_id].copy()
        if subset.empty:
            row_data[f'AssetID_{other_id}'] = np.nan
            continue

        subset['ts_diff'] = np.abs(subset['TimeStamp'] - main_ts)
        close_rows = subset[subset['ts_diff'] <= 1800]

        if not close_rows.empty:
            closest_row = close_rows.sort_values('ts_diff').iloc[0]
            row_data[f'AssetID_{other_id}'] = closest_row['Value']
            used_indices.append(closest_row.name)
        else:
            row_data[f'AssetID_{other_id}'] = np.nan

    # مرحله 7: حذف ردیف‌های استفاده‌شده
    df.drop(index=used_indices, inplace=True)

    # مرحله 8: اضافه کردن ردیف جدید به خروجی
    output_df = pd.concat([output_df, pd.DataFrame([row_data])], ignore_index=True)

# ============= بخش اضافه شده برای حذف ردیف‌های دارای NaN =============
# فقط ردیف‌هایی را نگه دار که تمام ستون‌های AssetID_* مقدار غیر NaN داشته باشند
asset_columns = [col for col in output_df.columns if col.startswith('AssetID_')]
output_df = output_df.dropna(subset=asset_columns, how='any')
# ====================================================================

# مرحله نهایی: تبدیل TimeStamp به تاریخ و زمان و اضافه کردن ستون‌های جدید
output_df['date'] = pd.to_datetime(output_df['TimeStamp'], unit='s')

# ایجاد ستون‌های RecordDate و RecordTime از ستون date
output_df['RecordDate'] = output_df['date'].dt.date
output_df['RecordTime'] = output_df['date'].dt.time

# اضافه کردن ستون id (شماره ردیف)
output_df.insert(0, 'id', range(1, len(output_df) + 1))

# حذف ستون TimeStamp
output_df.drop(columns=['TimeStamp'], inplace=True)

# مرتب‌سازی بر اساس تاریخ
output_df.sort_values(by='date', inplace=True)

# ذخیره خروجی در فایل اکسل
output_df.to_excel(output_file, index=False)

print("✅ پردازش کامل شد. ستون‌های زیر اضافه شدند:")
print("1. id (شماره ردیف)")
print("2. date (تاریخ و زمان کامل)")
print("3. RecordDate (تاریخ)")
print("4. RecordTime (زمان)")
print(f"✅ تعداد ردیف‌های نهایی (بدون NaN): {len(output_df)}")
print("✅ خروجی ذخیره شد.")

✅ پردازش کامل شد. ستون‌های زیر اضافه شدند:
1. id (شماره ردیف)
2. date (تاریخ و زمان کامل)
3. RecordDate (تاریخ)
4. RecordTime (زمان)
✅ تعداد ردیف‌های نهایی (بدون NaN): 11752
✅ خروجی ذخیره شد.
